# AORUS MASTER 16 AM6H Spec RAG — T4 benchmark

Everything in the repository runs on Apple Silicon during development, but Metal
uses unified memory and therefore cannot demonstrate the assignment's **4GB VRAM**
limit. This notebook exists to produce the numbers the README reports:

1. **VRAM evidence** — what the running system actually occupies on a discrete GPU.
2. **TTFT and TPS** on that GPU.
3. The retrieval and generation evaluations, reproduced end to end from a clean clone.

Runtime → Change runtime type → **T4 GPU** before running anything.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

## 2. Clone the repository

In [ ]:
!git clone --depth 1 https://github.com/maxxyhc/Gigabyte.git /content/repo
%cd /content/repo
!ls

## 3. Environment via `uv`

`uv sync --frozen` installs exactly the versions in `uv.lock`, so this is also
the check that the committed lockfile reproduces the development environment.

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

import os
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]

!uv sync --frozen

## 4. Build llama.cpp with CUDA

The project's releases ship no prebuilt Linux CUDA binary, so the server is built
from source. `CMAKE_CUDA_ARCHITECTURES=native` compiles for this runtime's GPU
only, which cuts the build from tens of minutes to a few.

In [ ]:
!apt-get -qq install -y cmake ninja-build 2>&1 | tail -1
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp

!cmake -S /content/llama.cpp -B /content/llama.cpp/build -G Ninja \
    -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON \
    -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF 2>&1 | tail -3
!cmake --build /content/llama.cpp/build --target llama-server -j 2>&1 | tail -3

!/content/llama.cpp/build/bin/llama-server --version

## 5. Download the quantised model (~2.3 GB)

In [ ]:
!mkdir -p models
!curl -L --progress-bar -o models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf \
    https://huggingface.co/unsloth/Qwen3-4B-Instruct-2507-GGUF/resolve/main/Qwen3-4B-Instruct-2507-Q4_K_M.gguf
!ls -lh models/

## 6. Build the vector index — on CPU

`embed.py` pins the encoder to CPU. The index is 21 vectors built once to a
`.npy`, and at query time only the question is encoded, which is imperceptible
off-GPU. Watch the VRAM reading in the next cell: this step contributes nothing
to it, which is what frees the whole budget for the LLM.

In [ ]:
!uv run python src/embed.py

## 7. Start llama-server and measure VRAM

`baseline` is what the GPU holds before the server starts, so the difference is
attributable to this workload rather than to whatever else the runtime is doing.

In [ ]:
import subprocess, time, requests

MODEL = "models/Qwen3-4B-Instruct-2507-Q4_K_M.gguf"

def gpu_used_mib() -> int:
    out = subprocess.run(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
        capture_output=True, text=True,
    )
    return int(out.stdout.split()[0])

baseline = gpu_used_mib()

server = subprocess.Popen(
    ["/content/llama.cpp/build/bin/llama-server",
     "-m", MODEL, "--host", "127.0.0.1", "--port", "8080",
     "--ctx-size", "4096", "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
     "-ngl", "99", "--jinja"],
    stdout=open("/content/server.log", "wb"), stderr=subprocess.STDOUT,
)

for _ in range(180):
    try:
        if requests.get("http://127.0.0.1:8080/health", timeout=2).json().get("status") == "ok":
            break
    except Exception:
        pass
    time.sleep(1)

loaded = gpu_used_mib()
print(f"baseline before server : {baseline:>6} MiB")
print(f"after model load       : {loaded:>6} MiB   (+{loaded - baseline} MiB)")

## 8. VRAM under load

The KV cache grows with the tokens actually processed, so the honest figure is
taken after real traffic, not straight after loading. This cell fills the context
with the longest prompt the pipeline produces and re-reads the meter.

In [ ]:
# Run through `uv run`, not this kernel: the project's dependencies live in the
# uv venv, and importing src/ here would need them installed a second time.
!uv run python src/rag.py "這台的顯卡、螢幕、連接埠和記憶體規格分別是什麼？" --max-tokens 320 --sources

under_load = gpu_used_mib()
print(f"\nunder load             : {under_load:>6} MiB   (+{under_load - baseline} MiB over baseline)")
print()
print(subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv"],
    capture_output=True, text=True).stdout)
print(f"4096 MiB budget: {'WITHIN' if under_load - baseline < 4096 else 'EXCEEDED'}")

## 9. Retrieval evaluation

No GPU involved — this scores the hybrid retriever against the 30-question golden
set and should reproduce the development numbers exactly, since nothing here is
sampled.

In [ ]:
!uv run python eval/run_eval.py 2>/dev/null

## 10. Generation evaluation

90 generations (30 questions × 3 configurations) at `temperature=0`. The TTFT and
TPS columns here are the ones the README reports; the Apple Silicon numbers are
kept only as a comparison.

In [ ]:
!uv run python eval/run_gen_eval.py 2>/dev/null

## 11. Final VRAM reading

In [ ]:
peak = gpu_used_mib()
print(f"after the full evaluation: {peak} MiB used, {peak - baseline} MiB attributable to this workload")
!nvidia-smi

## 12. Shut down

Frees the GPU so the reading above is the last word on what the workload held.

In [ ]:
server.terminate()
server.wait(timeout=30)
print("stopped")